In [40]:
import numpy as np
import pandas as pd
import yfinance as yf
import quantstats as qs
from datetime import datetime

# Import Coins Data

In [9]:
coins = ("BTC-USD", "ETH-USD", "XRP-USD", "SOL-USD", "DOGE-USD")

start = datetime(2024, 1, 1)
end = datetime(2024, 11, 1)

In [10]:
price_dict = {}
for coin in coins:
    data = yf.Ticker(coin).history(interval="1d", start=start, end=end)
    price_dict[coin] = data.Close

prices_df = pd.DataFrame(price_dict)

# Buy and Hold Strategy Backtest and Measurements

In [30]:
def first_price(series: pd.DataFrame, coin: str) -> float:
    return series.iloc[0][coin]

In [31]:
def get_share(coin: str, fund: float, price: float, weight: float) -> float:
    return (fund * weight) / price

In [38]:
def backtest(series: pd.DataFrame, fund: int, weights: dict) -> pd.Series:
    initial_prices = {coin: first_price(series, coin) for coin in series.columns}
    shares = {
        coin: get_share(coin, fund, initial_prices[coin], weights[coin])
        for coin in series.columns
    }
    portfolio_daily = pd.DataFrame([None for _ in series.index], index=series.index)
    portfolio_daily.columns = ["Value"]
    coins = series.columns
    fee = 0.5
    for day in series.index:
        value = 0
        for coin in coins:
            value += shares[coin] * series.loc[day, coin]
        portfolio_daily.loc[day, "Value"] = value * (100 + fee) / 100
        fee = 0
    return portfolio_daily

In [34]:
def generate_random_weight(coins: tuple) -> dict:
    weights = np.random.rand(len(coins))
    weights /= np.sum(weights)
    weights = {coin: weights[i] for i, coin in enumerate(coins)}
    return weights

In [35]:
random_weights = generate_random_weight(coins)

In [41]:
result = backtest(prices_df, 1000, random_weights)

In [50]:
text = f'''Sharpe Ratio is : {qs.stats.sharpe(result)}
Expected Return is : {qs.stats.expected_return(result)}
Maximum Drawdown is : {qs.stats.max_drawdown(result)}
Risk(Variance is : {qs.stats.volatility(result)}
'''

In [51]:
print(text)

Sharpe Ratio is : Value    0.707846
dtype: float64
Expected Return is : Value    0.000902
dtype: float64
Maximum Drawdown is : Value   -0.336287
dtype: float64
Risk(Variance is : Value    0.49157
dtype: float64

